# 01 – EDA & pressure exploration
Loads the processed + scored tables. Run `scripts/build_dataset.py` and `scripts/train_baselines.py` first.

In [ ]:
import sys, pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
sys.path.insert(0, '../src')
from rr_pressure.paths import PROCESSED, FIGURES
pd.set_option('display.max_columns', 80)

In [ ]:
pr = pd.read_parquet(PROCESSED / 'provided_deliveries_scored.parquet')
cs = pd.read_parquet(PROCESSED / 'cricsheet_deliveries.parquet')
pmap = pd.read_csv(PROCESSED / 'player_id_map.csv')
pr.shape, cs.shape, pmap.method.value_counts()

In [ ]:
pr.groupby(['match_format', 'season']).match_id.nunique().unstack(0)

## Leverage distribution by phase / innings

In [ ]:
sns.boxplot(data=pr, x='phase', y='leverage', hue='innings', showfliers=False); plt.title('Ball leverage by phase');

## Example pressure flag: top 20% leverage balls

In [ ]:
thr = pr.leverage.quantile(0.8)
pr['high_pressure'] = pr.leverage >= thr
bat = (pr[pr.is_legal | pr.is_noball].groupby(['batter', 'high_pressure'])
         .agg(balls=('is_legal', 'sum'), runs=('runs_batter', 'sum'), xruns=('xruns', 'sum'),
              outs=('is_wicket', 'sum'), wpa=('wpa', 'sum')).unstack())
bat[('balls', True)].sort_values(ascending=False).head(20)